# B1 · Maps, coordinates, latitude and longitude: placing Nepal on the Earth

> **Time:** about 90 minutes  
> **You need:** all of Part A. You will use functions and the `math` module (A5), pandas tables (A9) and scatter plots (A10).  
> **By the end you can:**
> 1. name the parts of a map and work with a map **scale** such as 1:50,000,
> 2. read **latitude** and **longitude**, convert between degrees-minutes-seconds and decimal degrees, and turn degrees into rough kilometres,
> 3. draw a first map of peaks and trek stops with longitude on the x-axis, and explain the difference between **vector** and **raster** data.

Welcome to Part B. In Part A, two columns in the course tables were different from the rest: `lat` and `lon`. They say *where* each record is. Part B is about those columns and what you can do with them.

This lesson lays the foundation for every map in the course. You will learn what a map is and how a position on the Earth is written down. Then you will draw one with the Matplotlib you already know.

**Counting lines:** as in Lesson A10, the lines `fig, ax = plt.subplots()` and `plt.show()` only set up a plot. They do not count toward the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 11 of 21 · [Course home](../README.md)  
← Previous: [A10 · Plotting with Matplotlib: charts of rain and mountains](../A_Python/A10_Plotting_with_Matplotlib.ipynb) · Next: [B2 · Points, lines and polygons: geometry with shapely](B02_Points_lines_and_polygons.ipynb) →

## 1. What a map is

A **map** is a simplified drawing of part of the Earth, made for a purpose. A trekking map shows trails and lodges; a geological map shows rock types. No map can show everything, so every map leaves things out on purpose.

Most maps share a few parts:

| Part | What it tells you |
| --- | --- |
| title | what the map shows, and where |
| **scale** | how distances on the map relate to distances on the ground |
| **legend** | what each colour and symbol means |
| **north arrow** | which way is north |
| source | where the data came from |

Every map is built from data that has positions. In the course data, the `lat` and `lon` columns hold those positions. The first cell loads everything this lesson needs.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

peaks = pd.read_csv(DATA / "nepal_peaks.csv")
trek = pd.read_csv(DATA / "everest_trek.csv")
trek

`trek` holds eight stops on the trek from Lukla to Everest Base Camp, in the order of the `order` column. The positions are approximate and the elevations are commonly cited values, good for teaching (see `data/README.md`).

### Practice

**Q1 · one line:** Show only the `name`, `lat` and `lon` columns of `peaks`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
peaks[["name", "lat", "lon"]]
```

These three columns are all a simple map needs: what each thing is, and where it is. Everest is at `lat` 27.9881 and `lon` 86.925.
</details>

**Q2 · two lines:** Print the number of rows in `trek`, then its column names.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(len(trek))
print(trek.columns)
```

There are 8 stops and 5 columns: `order`, `place`, `lat`, `lon` and `elevation_m`.
</details>

**Q3 · three lines:** A larger `lat` value means farther north (step 2 explains why). Sort `trek` by `lat`, largest first. Print the `place` and `lat` of the first row.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
northmost = trek.sort_values("lat", ascending=False)
print(northmost.iloc[0]["place"])
print(northmost.iloc[0]["lat"])
```

Everest Base Camp, at 28.0029, is the farthest north. That makes sense: the trail climbs north up the valley towards Everest.
</details>

## 2. The Earth as a grid: latitude and longitude

To give a position on the round Earth, we use two angles, measured from the centre of the Earth.

- **Latitude** says how far north or south a place is: 0° at the **equator**, 90° at each pole. Lines of equal latitude are called **parallels**; they run east–west.
- **Longitude** says how far east or west a place is: 0° at the **prime meridian** through Greenwich, England, up to 180°. Lines of equal longitude are called **meridians**; they run from pole to pole.

![Lines of latitude (left) and longitude (right) on a globe; the 0° lines are red](../images/lat_vs_lon.png)

*Image: the pyGIS project by Michael Mann (GPL-3.0).*

The diagram shows the Americas, but the grid is the same everywhere. Together the parallels and meridians form a **graticule**, the grid printed on many maps.

North and east are written as positive numbers, south and west as negative ones. Nepal is north of the equator and east of Greenwich, so both of its numbers are positive.

**Predict:** which of the eight peaks is farthest west, and which is farthest east?

In [ ]:
by_name = peaks.set_index("name")
print(f"Everest: {by_name.loc['Everest', 'lat']}° N, {by_name.loc['Everest', 'lon']}° E")

In [ ]:
print(peaks.sort_values("lon")[["name", "lat", "lon"]])

Sorting by longitude orders the peaks from west to east. Dhaulagiri I (83.4875° E) is farthest west and Kanchenjunga (88.1475° E) farthest east. `by_name` uses the peak names as row labels, as in Lesson A9, so `.loc` can find a peak by name.

### Practice

**Q1 · one line:** Use `by_name` to print Kanchenjunga's longitude.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(by_name.loc["Kanchenjunga", "lon"])
```

It prints 88.1475: Kanchenjunga is about 88° east of Greenwich.
</details>

**Q2 · two lines:** How many degrees of longitude separate Dhaulagiri I and Kanchenjunga? Store the difference, then print it rounded to 2 decimal places.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
span_deg = by_name.loc["Kanchenjunga", "lon"] - by_name.loc["Dhaulagiri I", "lon"]
print(round(span_deg, 2))
```

The two peaks are 4.66 degrees of longitude apart. Rounding hides the tiny binary leftovers you met in Lesson A1.
</details>

**Q3 · three lines:** Write a function `hemisphere(lat_deg)` that returns `"N"` for a latitude of 0 or more, and `"S"` otherwise (Lessons A4 and A5). Test it on Everest's latitude and on -13.5 (a made-up place south of the equator).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
def hemisphere(lat_deg):
    if lat_deg >= 0:
        return "N"
    return "S"

print(hemisphere(by_name.loc["Everest", "lat"]), hemisphere(-13.5))
```

It prints `N S`. The sign of the number carries the hemisphere, so a GIS stores no separate N or S letter. A `return` ends the function, so no `else` is needed.
</details>

## 3. Degrees, minutes and seconds

A degree can be split like an hour: **1 degree = 60 minutes** (′) and **1 minute = 60 seconds** (″). Old maps, GPS units and many websites write positions this way, for example 27° 59′ 17″ N. This is called **degrees-minutes-seconds** (DMS).

Computers prefer **decimal degrees**: one number, such as 27.9881.

- DMS to decimal: divide the minutes by 60 and the seconds by 3,600, then add everything to the degrees.
- Decimal to DMS: `int()` keeps the whole degrees. Multiply what is left by 60 to get minutes, and repeat to get seconds.

For a position south or west, convert the positive numbers, then put a minus sign in front of the result.

`return degrees, minutes, seconds` hands back three values at once. `d, m, s = decimal_to_dms(...)` unpacks them into three names, as `fig, ax = plt.subplots()` does.

**Predict:** what will `decimal_to_dms(27.9881)` give?

In [ ]:
def dms_to_decimal(degrees, minutes, seconds):
    """Convert degrees, minutes and seconds to decimal degrees."""
    return degrees + minutes / 60 + seconds / 3600


def decimal_to_dms(decimal_degrees):
    """Split decimal degrees into whole degrees, whole minutes and seconds."""
    degrees = int(decimal_degrees)
    minutes_float = (decimal_degrees - degrees) * 60
    minutes = int(minutes_float)
    seconds = (minutes_float - minutes) * 60
    return degrees, minutes, round(seconds, 1)


print(dms_to_decimal(27, 59, 17))
print(decimal_to_dms(27.9881))

27° 59′ 17″ is 27.98806 in decimal degrees, and 27.9881 is 27° 59′ 17.2″. The two differ by a fraction of a second: a few metres on the ground, as step 4 will show.

### Practice

**Q1 · one line:** Convert Everest's longitude, 86° 55′ 30″, to decimal degrees.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dms_to_decimal(86, 55, 30)
```

The result is 86.925 (Python may show a long tail of digits such as `86.92500000000001`). It matches the `lon` value in `nepal_peaks.csv`.
</details>

**Q2 · two lines:** Convert Lukla's latitude (the first row of `trek`) to DMS, and print it as text like `27° 41′ 11.8″`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
d, m, s = decimal_to_dms(trek["lat"].iloc[0])
print(f"{d}° {m}′ {s}″")
```

It prints `27° 41′ 11.8″`. The function returns three values, and the f-string puts each one next to its symbol.
</details>

**Q3 · three lines:** A GPS reading of a landslide says 27° 52′ 19.2″ N, 86° 9′ 2.2″ E (a made-up reading for practice). Convert both parts and print them rounded to 4 decimal places. Then look in `practice_landslides.csv`: which record is it?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
slide_lat = dms_to_decimal(27, 52, 19.2)
slide_lon = dms_to_decimal(86, 9, 2.2)
print(round(slide_lat, 4), round(slide_lon, 4))
```

It prints `27.872 86.1506`, the position of record LS001 in Dolakha. Rounding to 4 decimal places keeps a precision of about 10 m, which is plenty for a landslide.
</details>

## 4. How many kilometres is a degree?

Degrees are angles, not distances. To plan a trek or measure a landslide, you need kilometres.

- One degree of **latitude** is about **111 km** everywhere, because every meridian has the same length.
- One degree of **longitude** is 111 km at the equator, but shrinks towards the poles, where the meridians meet. The rule is: km per degree of longitude ≈ **111 × cos(latitude)**.

`math.cos` works in radians, like `math.tan` in Lesson A5, so convert first: `math.cos(math.radians(lat_deg))`.

These are approximations, good for short distances. Lesson B4 measures properly in metres.

**Predict:** at 60° N, how long is one degree of longitude?

In [ ]:
def km_per_degree_lon(lat_deg):
    """Approximate length in km of one degree of longitude at a given latitude."""
    return 111 * math.cos(math.radians(lat_deg))


for lat_deg in [0, 28, 60]:
    print(lat_deg, round(km_per_degree_lon(lat_deg), 1))

print(111000 / 3600)   # metres in one second of latitude

In [ ]:
# straight-line distance from Lukla to Everest Base Camp
lukla = trek.iloc[0]
base_camp = trek.iloc[-1]

north_km = (base_camp["lat"] - lukla["lat"]) * 111
east_km = (base_camp["lon"] - lukla["lon"]) * km_per_degree_lon(lukla["lat"])
straight_km = (north_km ** 2 + east_km ** 2) ** 0.5   # Pythagoras
print(round(north_km, 1), round(east_km, 1), round(straight_km, 1))

At 28° N, the latitude of Nepal, a degree of longitude is about 98 km. At 60° N it is only 55.5 km, half the length at the equator. One second of latitude is about 31 m, so the difference of 0.2″ in step 3 is about 6 m.

Base Camp is 35.1 km north and 12.1 km east of Lukla, so the straight line between them is about 37.1 km. Over such short distances the ground is nearly flat, and Pythagoras works well. The walking distance along the trail is longer.

### Practice

**Q1 · one line:** How many km is one degree of longitude at Everest's latitude, 27.9881°? Round to 1 decimal place.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(km_per_degree_lon(27.9881), 1)
```

About 98.0 km: almost the same as at 28°, because the latitude is almost the same.
</details>

**Q2 · two lines:** How many km north of Namche Bazaar (row 2 of `trek`) is Everest Base Camp? Use `base_camp` from the example.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
namche = trek.iloc[2]
print(round((base_camp["lat"] - namche["lat"]) * 111, 1))
```

Base Camp is about 22.0 km north of Namche Bazaar. Only latitude matters for a north–south distance, so the 111 km rule is enough.
</details>

**Q3 · three lines:** How many km apart are Dhaulagiri I and Kanchenjunga from east to west? Use `by_name` from step 2 and the length of a degree of longitude at 28° N.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
span_deg = by_name.loc["Kanchenjunga", "lon"] - by_name.loc["Dhaulagiri I", "lon"]
span_km = span_deg * km_per_degree_lon(28)
print(round(span_km, 1))
```

About 456.7 km. The eight peaks spread along most of the length of Nepal.
</details>

## 5. x is longitude: your first map

A simple map is a scatter plot with a special meaning.

Longitude grows from west to east, as the x-axis grows from left to right. Latitude grows from south to north, as the y-axis grows upwards. So **x = longitude** and **y = latitude**, and north is at the top.

We *say* "latitude and longitude", but code almost always wants **(x, y) = (lon, lat)**. Swapping them is one of the most common bugs in GIS.

Near Nepal a degree of longitude is shorter than a degree of latitude (step 4). `ax.set_aspect(1 / math.cos(math.radians(28)))` draws each degree of latitude that much taller, so shapes are not squashed.

**Predict:** in the right panel, lat and lon are swapped. Which way will the trail run?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(trek["lon"], trek["lat"], marker="o")
ax1.set_title("Right: x = lon, y = lat")
ax1.set_xlabel("Longitude (°E)")
ax1.set_ylabel("Latitude (°N)")

ax2.plot(trek["lat"], trek["lon"], marker="o", color="tab:red")
ax2.set_title("Wrong: x = lat, y = lon")
ax2.set_xlabel("Latitude (°N) on the x-axis!")
ax2.set_ylabel("Longitude (°E) on the y-axis!")

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(peaks["lon"], peaks["lat"], marker="^", color="tab:red", label="Peaks above 8,000 m")
ax.plot(trek["lon"], trek["lat"], color="tab:blue", label="Lukla to Everest Base Camp")
for i in range(len(peaks)):
    row = peaks.iloc[i]
    ax.text(row["lon"] + 0.05, row["lat"] + 0.05, row["name"], fontsize=8)

ax.set_aspect(1 / math.cos(math.radians(28)))
ax.set_title("Nepal's eight peaks above 8,000 m")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.legend(loc="lower left")
plt.show()

In the left panel the trail climbs north (up), as it does on the ground. In the right panel it runs from left to right instead of upwards: north has become east. Nothing crashed, which is what makes this bug dangerous. Always check that a map "looks right".

In the second map, the eight peaks fall in a line from west to east, along the Himalaya. At this scale the whole trek is a short blue line near Everest. Everest and Lhotse are so close that their labels overlap; Lesson B11 shows how to place labels well.

### Practice

**Q1 · one line:** Draw a scatter plot of the eight trek stops, with longitude on the x-axis.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.scatter(trek["lon"], trek["lat"])
plt.show()
```

One line of real work: `ax.scatter(x, y)` with x = longitude. The stops form a curve running north.
</details>

**Q2 · two lines:** Draw the trek as a line with a dot at each stop. Set the aspect so that a degree of latitude is drawn correctly for 28° N.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
ax.plot(trek["lon"], trek["lat"], marker="o")
ax.set_aspect(1 / math.cos(math.radians(28)))
plt.show()
```

With the aspect set, one degree north is drawn about 1.13 times longer than one degree east, as on the ground. The trail looks narrower and taller than before.
</details>

**Q3 · three lines:** Draw the trek line again and label every stop with its `place` name, using a loop with `ax.text` as in Lesson A10.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(trek["lon"], trek["lat"], marker="o")
for i in range(len(trek)):
    row = trek.iloc[i]
    ax.text(row["lon"] + 0.003, row["lat"], row["place"])
plt.show()
```

Each label is placed at its stop's (lon, lat), moved a little to the right. You can follow the trail from Lukla in the south to Everest Base Camp in the north.
</details>

## 6. Map scale

A map's **scale** is the ratio between a distance on the map and the same distance on the ground. **1:50,000** means that 1 cm on the map is 50,000 cm on the ground, which is 500 m.

- A **large-scale** map, such as 1:25,000, shows a small area in great detail.
- A **small-scale** map, such as 1:1,000,000, shows a large area with little detail.

The words feel backwards, but think of the fraction: 1/25,000 is larger than 1/1,000,000.

Many maps also draw a **scale bar**, a small ruler marked in km. A scale bar stays correct when a map is enlarged on a screen; a printed ratio such as 1:50,000 does not.

Remember the units: 1 m = 100 cm, and 1 km = 100,000 cm.

In [ ]:
def map_cm_to_ground_km(map_cm, scale):
    """Ground distance in km for a distance measured on a map of scale 1:scale."""
    return map_cm * scale / 100000


def ground_km_to_map_cm(ground_km, scale):
    """Map distance in cm for a ground distance, on a map of scale 1:scale."""
    return ground_km * 100000 / scale


print(map_cm_to_ground_km(4, 50000))
print(round(ground_km_to_map_cm(straight_km, 50000), 1))

4 cm on a 1:50,000 map is 2 km on the ground, as in the last review of Lesson A1. The 37.1 km straight line from Lukla to Base Camp would be 74.3 cm long at 1:50,000: too long for one sheet of paper. That is why large-scale maps come as many separate sheets.

### Practice

**Q1 · one line:** Two springs are 7 cm apart on a 1:25,000 map. How far apart are they on the ground?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
map_cm_to_ground_km(7, 25000)
```

1.75 km. At 1:25,000, each centimetre on the map is 250 m on the ground.
</details>

**Q2 · two lines:** How long would the Lukla to Base Camp line (`straight_km`) be on a 1:100,000 map and on a 1:250,000 map? Print both, rounded to 1 decimal place.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(ground_km_to_map_cm(straight_km, 100000), 1))
print(round(ground_km_to_map_cm(straight_km, 250000), 1))
```

37.1 cm and 14.9 cm. The larger the number after the colon, the smaller the map: 1:250,000 is a smaller scale than 1:100,000.
</details>

**Q3 · three lines:** You must fit the 456.7 km between Dhaulagiri I and Kanchenjunga (step 4) across a page 30 cm wide. Work out the scale number you need: ground distance in cm divided by map distance in cm. Print it as a whole number.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ground_cm = 456.7 * 100000
scale_needed = ground_cm / 30
print(round(scale_needed))
```

You need about 1:1,522,333. In practice you would round up to a tidy scale such as 1:2,000,000, which leaves a margin. Rounding *up* makes the map slightly smaller, so it still fits.
</details>

## 7. Vector and raster: features with attributes

GIS stores geography in two main ways.

- **Vector** data stores things as exact shapes made of coordinates: **points** (a peak), **lines** (a trail, a river) and **polygons** (a district, a lake).
- **Raster** data stores a grid of cells, each holding one value, like the made-up elevation grid in Lesson A8 or a satellite image.

![The same river stored as vector data (left) and as raster data (right)](../images/vector_vs_raster.jpg)

*Image: the pyGIS project by Michael Mann (GPL-3.0).*

One thing in the world, stored in a vector dataset, is called a **feature**. Each feature has a **geometry** (its shape and position) and **attributes** (facts about it), such as `height_m` and `first_ascent`. So a vector dataset is a table with one row per feature, plus geometry.

The map's scale decides the geometry: Kathmandu can be a point on a map of Asia and a polygon on a map of the valley.

The next cell is a preview of Lesson B3. It reads the boundaries of Nepal's 77 districts, which are polygons, and draws your points on top.

In [ ]:
import geopandas as gpd

districts = gpd.read_file(DATA / "nepal_districts.geojson")
print(len(districts))
print(districts.total_bounds)

fig, ax = plt.subplots(figsize=(10, 5))
districts.boundary.plot(ax=ax, color="grey", linewidth=0.5)
ax.scatter(peaks["lon"], peaks["lat"], marker="^", color="tab:red", label="Peaks above 8,000 m")
ax.plot(trek["lon"], trek["lat"], color="tab:blue", label="Lukla to Everest Base Camp")
ax.set_aspect(1 / math.cos(math.radians(28)))
ax.set_title("Peaks and the Everest trek on Nepal's district boundaries")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.legend(loc="lower left")
plt.show()

GeoPandas read 77 district polygons, and `.boundary.plot` drew their outlines. The peaks sit along the northern border, in the right places, because the points and the polygons both use longitude and latitude.

`total_bounds` gives the box around all the districts: from about 80.06° to 88.20° E and from 26.35° to 30.45° N. Nepal lies inside that box. Lesson B3 explains every line of this cell.

### Practice

**Q1 · one line:** How many point features are in `trek`?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
len(trek)
```

8 features: one row per trek stop. Each has a geometry (its `lat` and `lon`) and attributes (`order`, `place`, `elevation_m`).
</details>

**Q2 · two lines:** Add a new attribute to the trek stops: a column `elevation_ft` (1 m = 3.28084 ft), rounded to whole feet. Show the `place` and `elevation_ft` columns.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
trek["elevation_ft"] = (trek["elevation_m"] * 3.28084).round()
trek[["place", "elevation_ft"]]
```

Adding a column adds an attribute to every feature at once, as in Lesson A9. Everest Base Camp is about 17,598 ft.
</details>

**Q3 · three lines:** A raster in miniature: make a 3 × 4 NumPy array of elevations in metres (made-up values for practice, Lesson A8). Print its shape and the value in row 1, column 2.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
import numpy as np
elev_m = np.array([[2860, 2900, 3000, 3100], [2950, 3050, 3200, 3300], [3100, 3250, 3440, 3600]])
print(elev_m.shape, elev_m[1, 2])
```

It prints `(3, 4) 3200`. A raster has a value in *every* cell, while vector data only stores the features themselves. Rasters are the subject of Part C.
</details>

## 8. When it breaks: coordinates written as text

You copy Everest's latitude from a website and try to turn it into a number. Run the cell.

In [ ]:
lat_text = "27° 59′ 17″"
float(lat_text)

Read the last line:

`ValueError: could not convert string to float: '27° 59′ 17″'`

A `ValueError` (Lesson A6) means the value had the right type, text, but Python could not read it as a number. `float()` understands decimal numbers only, such as `"27.9881"`. The symbols °, ′ and ″, the spaces and any N or E letters are not part of a number.

The fix: type the three parts as numbers and convert them with your function, `dms_to_decimal(27, 59, 17)`.

When a whole column arrives like this, pandas reads it as text. Check `df.dtypes` (Lesson A9) whenever you load coordinates: `lat` and `lon` must be `float64`.

**Try it:** replace the second line with the `dms_to_decimal` call.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · From a field notebook to decimal degrees

A field notebook lists three trek stops in DMS. Namche Bazaar: 27° 48′ 16.9″ N, 86° 42′ 50.0″ E. Tengboche: 27° 50′ 10.3″ N, 86° 45′ 52.6″ E. Dingboche: 27° 53′ 31.2″ N, 86° 49′ 51.6″ E. Store them as a list of dictionaries (Lesson A4) with the keys `place`, `lat_dms` and `lon_dms`, where each DMS value is a list of three numbers. Loop over the records and print each place with its decimal latitude and longitude, to 4 decimal places.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
stops = [
    {"place": "Namche Bazaar", "lat_dms": [27, 48, 16.9], "lon_dms": [86, 42, 50.0]},
    {"place": "Tengboche", "lat_dms": [27, 50, 10.3], "lon_dms": [86, 45, 52.6]},
    {"place": "Dingboche", "lat_dms": [27, 53, 31.2], "lon_dms": [86, 49, 51.6]},
]
for stop in stops:
    la = stop["lat_dms"]
    lo = stop["lon_dms"]
    lat_dd = dms_to_decimal(la[0], la[1], la[2])
    lon_dd = dms_to_decimal(lo[0], lo[1], lo[2])
    print(f"{stop['place']}: {lat_dd:.4f}, {lon_dd:.4f}")
```

Namche Bazaar comes out at 27.8047, 86.7139, exactly the values in `everest_trek.csv`; the other two match as well. Each list is unpacked by position: degrees, minutes, seconds.
</details>

### Review 2 · How far is each leg of the trek?

For each pair of neighbouring stops in `trek`, work out the straight-line distance in km. Use 111 km per degree of latitude, and `km_per_degree_lon` at the first stop's latitude. Print each leg, rounded to 1 decimal place, and the total at the end.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
total_km = 0
for i in range(len(trek) - 1):
    a = trek.iloc[i]
    b = trek.iloc[i + 1]
    north_km = (b["lat"] - a["lat"]) * 111
    east_km = (b["lon"] - a["lon"]) * km_per_degree_lon(a["lat"])
    leg_km = (north_km ** 2 + east_km ** 2) ** 0.5
    total_km += leg_km
    print(f"{a['place']} to {b['place']}: {leg_km:.1f} km")
print(f"Total: {total_km:.1f} km")
```

The legs range from 3.4 km (Gorak Shep to Base Camp) to 9.0 km (Tengboche to Dingboche), 42.4 km in total. That is longer than the 37.1 km straight line, because the trail bends. A negative difference (a leg going west or south) is squared, so it still adds a positive distance.
</details>

### Review 3 · A labelled map of the eight peaks

Draw a map of the eight peaks as red triangles. Label each one with its name and height, for example `Everest 8849 m`. Add the correct aspect for 28° N, a title, axis labels with units and a legend. Save it as `b1_peaks_map.png` in `OUT` at 300 dpi, then show it.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(peaks["lon"], peaks["lat"], marker="^", s=60, color="tab:red", label="Peak above 8,000 m")
for i in range(len(peaks)):
    row = peaks.iloc[i]
    label = f"{row['name']} {row['height_m']} m"
    if row["name"] == "Lhotse":     # put Lhotse's label on the left, away from Everest's
        ax.text(row["lon"] - 0.05, row["lat"] - 0.05, label, fontsize=8, ha="right")
    else:
        ax.text(row["lon"] + 0.05, row["lat"], label, fontsize=8)
ax.set_aspect(1 / math.cos(math.radians(28)))
ax.set_title("Nepal's peaks above 8,000 m")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.legend(loc="lower left")
fig.savefig(OUT / "b1_peaks_map.png", dpi=300, bbox_inches="tight")
plt.show()
```

`s=60` makes the markers larger. Everest and Lhotse are so close that their labels would overlap, so an `if` (Lesson A4) puts Lhotse's label on the left. `ha="right"` (horizontal alignment) makes the text *end* at the given x position instead of starting there. The labels include the height, so the map carries attributes as well as positions. Save before `plt.show()`, as in Lesson A10.
</details>

### Review 4 · Which scale fits the trek on a page?

You want to print the trek on a page whose map area is 18 cm wide and 25 cm tall. Work out the trek's north–south extent in km, from the smallest to the largest `lat`. Then work out its east–west extent, from the smallest to the largest `lon`, at the trek's mean latitude. Then work out the scale number needed for each direction, and print the larger one. Why the larger one?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
ns_km = (trek["lat"].max() - trek["lat"].min()) * 111
ew_km = (trek["lon"].max() - trek["lon"].min()) * km_per_degree_lon(trek["lat"].mean())
print(round(ns_km, 1), round(ew_km, 1))

scale_ns = ns_km * 100000 / 25    # the north–south extent must fit 25 cm
scale_ew = ew_km * 100000 / 18    # the east–west extent must fit 18 cm
print(round(scale_ns), round(scale_ew))
print(round(max(scale_ns, scale_ew)))
```

The trek spans 35.1 km north–south and 13.7 km east–west. It needs 1:140,437 to fit the height and 1:76,217 to fit the width. The larger number gives the smaller map, so only 1:140,437 fits in both directions; round it up to 1:150,000.
</details>

## Recap

- A **map** is a simplified drawing of part of the Earth, made for a purpose, with a title, scale, legend, north arrow and source.
- **Latitude** measures north–south from the equator, **longitude** east–west from Greenwich. North and east are positive. DMS converts to decimal degrees with d + m/60 + s/3600.
- One degree of latitude is about 111 km; one degree of longitude is about 111 × cos(latitude) km, about 98 km in Nepal.
- In code, **x = longitude** and **y = latitude**. A swapped map does not crash, so always look at it.
- A scale of 1:50,000 means 1 cm is 500 m. **Vector** data stores features (points, lines, polygons) with attributes; **raster** data stores a grid of values.

### Words you learned

| Word | Meaning |
| --- | --- |
| map | A simplified drawing of part of the Earth, made for a purpose |
| scale | The ratio between a distance on the map and on the ground, such as 1:50,000 |
| large-scale / small-scale | Large-scale (1:25,000) shows a small area in detail; small-scale (1:1,000,000) shows a large area |
| scale bar | A small ruler on a map, marked in ground units such as km |
| legend | The key that explains a map's colours and symbols |
| north arrow | A symbol showing which way is north |
| latitude | The angle north (+) or south (−) of the equator, from 0° to 90° |
| longitude | The angle east (+) or west (−) of the prime meridian, from 0° to 180° |
| equator | The line of 0° latitude, halfway between the poles |
| prime meridian | The line of 0° longitude, through Greenwich, England |
| parallel / meridian | A line of equal latitude / a line of equal longitude |
| graticule | The grid of parallels and meridians drawn on a map |
| DMS | Degrees, minutes (′) and seconds (″); 1° = 60′ and 1′ = 60″ |
| decimal degrees | A position as one decimal number of degrees, such as 27.9881 |
| aspect | The ratio between the drawn size of one y unit and one x unit |
| vector data | Geography stored as points, lines and polygons made of coordinates |
| raster data | Geography stored as a grid of cells, each with one value |
| feature | One thing in a vector dataset, such as one peak or one district |
| geometry | The shape and position of a feature |
| attribute | A fact about a feature, stored in a column, such as `height_m` |

## Next

**B2 · Points, lines and polygons:** build points, lines and polygons in Python with the shapely library. Everest's summit, the whole trek and a box around the Kathmandu Valley become shapes you can measure and compare.

<!-- pygile-nav -->
**PyGILE Course** · lesson 11 of 21 · [Course home](../README.md)  
← Previous: [A10 · Plotting with Matplotlib: charts of rain and mountains](../A_Python/A10_Plotting_with_Matplotlib.ipynb) · Next: [B2 · Points, lines and polygons: geometry with shapely](B02_Points_lines_and_polygons.ipynb) →